In [1]:
# Relevant packages for Stanford data visualization
import open3d as o3d
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import time
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [62]:
def apply_colormap(data, transform='linear'):
    if transform == 'log':
        norm = mcolors.LogNorm()(data)
    elif transform == 'power':
        norm = data**3  # Using a power of 0.3 as an example
    else:
        norm = data
    colors = plt.cm.RdBu(norm)
    return colors

import pandas as pd
import plotly.express as px


def plot_vs(points, visibility, viewpoint, out_file=None):
    df_3d = pd.DataFrame({'x': points[:, 0], 'y': points[:, 1], 'z': points[:, 2], 'visibility': visibility})
    fig = px.scatter_3d(df_3d, x='x', y='y', z='z', color='visibility', title='Visibility Score for Stanford Bunny',
                        color_continuous_scale=['red', 'blue'])
    fig.update_traces(marker=dict(size=1))  # Adjust the size value as needed
    fig.update_layout(
        autosize=False,
        width=500,
        height=500,
        margin=dict(
            l=0,  # left margin
            r=0,  # right margin
            b=0,  # bottom margin
            t=0,  # top margin
            pad=0  # padding
        ),
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            camera=dict(
                up=dict(x=0, y=1, z=0),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=viewpoint[0], y=viewpoint[1], z=viewpoint[2])  # Adjust the eye position here
            )
        ),
        showlegend=False,
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(0,0,0,0)',
    )

    fig.show()
    if out_file is not None:
        pio.write_image(fig, out_file)


def plot_points(points, viewpoint, plot_viewpoint=True, 
                points_special_inds=None, extra_viewpoint=None, out_file="output.eps",
                save=True, visibility_scores=None):

    # If visibility scores are provided, use them to determine colors
    if visibility_scores is not None:
        # colorscale = [[0, 'red'], [1, 'blue']]
        # colors = np.interp(visibility_scores, (visibility_scores.min(), visibility_scores.max()), [0, 1])
        #colors = apply_colormap(visibility_scores)
        colors = visibility_scores
        # colors_sorted = plt.cm.RdBu(np.linspace(0, 1, len(visibility_scores)))
        # # Sorting the torch tensor and getting the indices
        # sorted_indices = torch.argsort(visibility_scores)

        # # Using the sorted colors to color the original data points
        # colors = colors_sorted[sorted_indices.numpy()]
    else:
        colors = 'blue'

    # Prepare point clouds
    if points_special_inds is not None:
        scatter_points_special = go.Scatter3d(
            x=points[points_special_inds, 0], 
            y=points[points_special_inds, 1], 
            z=points[points_special_inds, 2], 
            mode='markers',
            marker=dict(
                size=0.7,
                opacity=1,
                color='green'  # Keep special points as green for now
            )
        )

        scatter_points_normal = go.Scatter3d(
            x=np.delete(points[:, 0], points_special_inds), 
            y=np.delete(points[:, 1], points_special_inds), 
            z=np.delete(points[:, 2], points_special_inds), 
            mode='markers',
            marker=dict(
                size=0.7,
                opacity=1,
                color=colors
            )
        )
    else:
        scatter_points_normal = go.Scatter3d(
            x=points[:, 0], 
            y=points[:, 1], 
            z=points[:, 2], 
            mode='markers',
            marker=dict(
                size=0.7,
                opacity=1,
                color=colors
            )
        )
    # Store point clouds
    data = [scatter_points_normal]

    if points_special_inds is not None:
        data.append(scatter_points_special)

    # Create a Scatter3d object for the viewpoint
    if plot_viewpoint:
        scatter_viewpoint = go.Scatter3d(
            x=[viewpoint[0]], 
            y=[viewpoint[1]], 
            z=[viewpoint[2]], 
            mode='markers',
            marker=dict(
                size=10,
                color='blue',
                opacity=1
            )
        )
        data.append(scatter_viewpoint)

    if extra_viewpoint is not None:
        scatter_extra_viewpoint = go.Scatter3d(
            x=[extra_viewpoint[0]], 
            y=[extra_viewpoint[1]], 
            z=[extra_viewpoint[2]], 
            mode='markers',
            marker=dict(
                size=10,
                color='yellow',
                opacity=1
            )
        )
        data.append(scatter_extra_viewpoint)
        
    fig = go.Figure(data=data)


    # Update layout to remove background and axes
    fig.update_layout(
        autosize=False,
        width=500,
        height=500,
        margin=dict(
            l=0,  # left margin
            r=0,  # right margin
            b=0,  # bottom margin
            t=0,  # top margin
            pad=0  # padding
        ),
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            camera=dict(
                up=dict(x=0, y=1, z=0),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=viewpoint[0], y=viewpoint[1], z=viewpoint[2])  # Adjust the eye position here
            )
        ),
        showlegend=False,
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(0,0,0,0)',
    )
    # import random
    # eye_x = random.uniform(-1, 1)
    # eye_y = random.uniform(-1, 1)
    # eye_z = random.uniform(-1, 1)
    
    # # Update the figure layout
    # fig.update_layout(scene_camera=dict(eye=dict(x=eye_x, y=eye_y, z=eye_z)))
    
    # # Show the plot
    fig.show()
    if save:
        pio.write_image(fig, out_file)

In [65]:
import open3d as o3d

pcd = o3d.io.read_point_cloud("data/stanford/bunny/reconstruction/bun_zipper.ply")
import time
import numpy as np
import importlib
import utils.visibility
importlib.reload(utils.visibility)
from utils.visibility import visible_points


points = np.asarray(pcd.points)
radius = 3.25
viewpoint = np.array([0, 0, -1.55])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
points = torch.tensor(points, device=device)
viewpoint = torch.tensor(viewpoint, device=device)

start = time.time()
visible_inds = visible_points(points, viewpoint, hpr_radius=radius, compute_weights=False)
end = time.time()
print(f"Time taken without vs: {np.around(end-start, 3)}")

# start = time.time()
# visible_inds, visibility_scores = visible_points(points, viewpoint, hpr_radius=radius, compute_weights=True)
# end = time.time()
# visibility_scores = visibility_scores - visibility_scores.min()
# visibility_scores = visibility_scores/visibility_scores.max()
# visibility_scores = visibility_scores.cpu().numpy()
# viewpoint = viewpoint.cpu().numpy()
# print(f"Time taken with vs: {np.around(end-start, 3)}")

# points_viewpoint = points[visible_inds].cpu().numpy()

# folder = 'images/point_clouds/stanford/'
# plot_vs(points_viewpoint, visibility_scores, viewpoint, out_file=folder + 'bunny_visibility_score_debug.eps')
# # plot_points(points_viewpoint, viewpoint, plot_viewpoint=False, out_file=folder + 'bunny_visibility_score.eps', visibility_scores=visibility_scores)
#plot_points(noisy_point_viewpoint, viewpoint, plot_viewpoint=False, out_file=folder + 'bunny_visibility_score.eps', visibility_scores=noisy_visibility_scores)

Time taken without vs: 0.086


In [ ]:

import utils.visibility
importlib.reload(utils.visibility)
from utils.visibility import visible_points

start = time.time()
visible_inds, visibility_scores = visible_points(points, viewpoint, hpr_radius=radius, compute_weights=True)
end = time.time()

In [66]:
%%timeit
visible_inds, visibility_scores = visible_points(points, viewpoint, hpr_radius=radius, compute_weights=True)

364 ms ± 1.28 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [79]:
print("hej dp")
pcd = o3d.io.read_point_cloud("data/stanford/bunny/reconstruction/bun_zipper.ply")
import utils.visibility
import importlib
importlib.reload(utils.visibility)
from utils.visibility import visible_points, covisible_inds


points = np.asarray(pcd.points)
n_points = points.shape[0]
radius = 3.25


viewpoint0 = np.array([0, 0, -1.55])
viewpoint1 = np.array([2, 0, -1.55])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
points = torch.tensor(points, device=device)
dtype = points.dtype
viewpoint0 = torch.tensor(viewpoint0, device=device, dtype=dtype)
viewpoint1 = torch.tensor(viewpoint1, device=device, dtype=dtype)



rot_hom = torch.vstack((torch.eye(3, device=device, dtype=dtype), torch.zeros(3, device=device, dtype=dtype)))
one = torch.ones(1, device=device, dtype=dtype)
T0 = torch.cat((rot_hom, torch.cat((viewpoint0, one))[:, None]), dim=1)
T1 = torch.cat((rot_hom, torch.cat((viewpoint1, one))[:, None]), dim=1)

visible_inds0 = visible_points(points, viewpoint0, hpr_radius=radius)
points_viewpoint0 = points[visible_inds0]
n_points0 = points_viewpoint0.shape[0]

visible_inds1 = visible_points(points, viewpoint1, hpr_radius=radius)
points_viewpoint1 = points[visible_inds1]
n_points1 = points_viewpoint1.shape[0]




points_permuted = torch.swapaxes(points_viewpoint0, axis0=1, axis1=0)
points_homog = torch.vstack((points_permuted, torch.ones(n_points0, device=device)))
pc0 = torch.swapaxes(torch.matmul(torch.inverse(T0), points_homog)[:3], axis0=1, axis1=0)

points_permuted = torch.swapaxes(points_viewpoint1, axis0=1, axis1=0)
points_homog = torch.vstack((points_permuted, torch.ones(n_points1, device=device, dtype=dtype)))
pc1_CS0 = torch.swapaxes(torch.matmul(torch.inverse(T0), points_homog)[:3], axis0=1, axis1=0)

pc_union = torch.vstack((pc0, pc1_CS0))

visible_inds, visible_inds_pc0, visible_inds_pc1 = covisible_inds(pc0, pc_union, T0, T1, hpr_radius=radius)
pc_union_cov = pc_union[visible_inds]
n_points1_cov = pc_union_cov.shape[0]

points_permuted = torch.swapaxes(pc_union_cov, axis0=1, axis1=0)
points_homog = torch.vstack((points_permuted, torch.ones(n_points1_cov, device=device)))
pc_union_cov_WCS = torch.swapaxes(torch.matmul(T0, points_homog)[:3], axis0=1, axis1=0)

pc_union_ground_truth_inds = np.intersect1d(visible_inds0, visible_inds1)
pc_union_ground_truth = points[pc_union_ground_truth_inds]
folder = 'images/point_clouds/stanford/'
plot_points(points_viewpoint0.cpu().numpy(), viewpoint0.cpu().numpy(), plot_viewpoint=False, out_file=folder + 'bunny_view0.eps')
plot_points(points_viewpoint1.cpu().numpy(), viewpoint1.cpu().numpy(), plot_viewpoint=False, out_file=folder + 'bunny_view1.eps')
plot_points(pc_union_ground_truth.cpu().numpy(), viewpoint0.cpu().numpy(), plot_viewpoint=False, out_file = folder + 'approx_gt_covisible.eps')
plot_points(pc_union_cov_WCS.cpu().numpy(), viewpoint0.cpu().numpy(), plot_viewpoint=False, out_file = folder + 'est_covisible.eps')

hej dp


In [ ]:
import nuscenes as ns

# Init Nusc object
data_folder = 'data/nuscenes/'
version = 'v1.0-mini'
nusc = ns.nuscenes.NuScenes(version=version, dataroot=data_folder, verbose=False)

In [ ]:
# Some visualization on visibility on the separate Nuscens point clouds

from utils.data_handling import split_data
from utils.nuscenes_handling import read_nuscenes_data
from utils.visibility import visible_points, covisible_inds
import torch
# Set params
N = 1
T_close_thresh = 1.5

#
PC_scenes = read_nuscenes_data(nusc, downsample_factor=2, n_scenes=N, n_samples=N, T_close_thresh=T_close_thresh)
PC_scenes_training, PC_scenes_test = split_data(PC_scenes, scenes_training=N)

for i in range(len(PC_scenes_training)):
    PC_pair = PC_scenes_training[i][0]
    zero_vec = np.zeros((3))
    T0 = PC_pair.pose0
    T1 = PC_pair.pose1
    viewpoint1_CS0 = torch.matmul(torch.linalg.inv(T0), T1).cpu().numpy()[:3, 3]
    pc_in = PC_pair.PC0.pc.cpu().numpy()

    #
    print(f"\nScene: {i}")
    for radius in [3.25]:
        visible_inds = visible_points(pc_in, zero_vec, hpr_radius=radius)
        visible_inds_other = visible_points(pc_in, viewpoint1_CS0, hpr_radius=radius)
        visible_inds, visible_inds_pc0, visible_inds_pc1 = covisible_inds(PC_pair.PC0, PC_pair.PCUnion, T0, T1)
        # print(f"Percentage visible points own viewpoint vs other " +
        #       f"{np.around(100*len(visible_inds)/pc_in.shape[0], 3)} % vs "+
        #       f"{np.around(100*len(visible_inds_other)/pc_in.shape[0], 3)} %")
        print(f"Covisible point percentage in union point cloud {np.around(100*visible_inds.shape[0]/PC_pair.PCUnion.N_points,2)} %")
#plot_points(pc_in, zero_vec, points_special_inds=visible_inds, extra_viewpoint=viewpoint1_CS0)
pc_union = PC_pair.PCUnion.pc.cpu().numpy()
# pc_union = pc_union[~visible_inds]
plot_points(np.delete(pc_union, visible_inds, axis=0), viewpoint=zero_vec, plot_viewpoint=False)
plot_points(pc_union[visible_inds], viewpoint=zero_vec, plot_viewpoint=False)


In [ ]:
# Some visualization on visibility on the joint Nuscens point clouds

from utils.data_handling import split_data
from utils.nuscenes_handling import read_nuscenes_data
from utils.visibility import covisible_inds, visible_points
import torch
import numpy as np
# Set params
N = 10
samples_per_scene = 10
T_close_thresh = 1.5
hpr_rads = np.arange(2, 5, 0.25)
#
PC_scenes = read_nuscenes_data(nusc, downsample_factor=1, n_scenes=N, n_samples=N*samples_per_scene, T_close_thresh=T_close_thresh)
PC_scenes_training, PC_scenes_test = split_data(PC_scenes, scenes_training=N)
# Initialize dictionaries to store the sum of percentages for each radius
sum_visible_perc_own = {rad: 0 for rad in hpr_rads}
sum_visible_perc_other = {rad: 0 for rad in hpr_rads}

for i in range(len(PC_scenes_training)):
    print(f"\nScene: {i}")
    for j in range(samples_per_scene):
      PC_pair = PC_scenes_training[i][j]
      zero_vec = np.zeros((3))
      T0 = PC_pair.pose0
      T1 = PC_pair.pose1
      viewpoint1_CS0 = torch.matmul(torch.linalg.inv(T0), T1).cpu().numpy()[:3, 3]
      PC_union = PC_pair.PCUnion.pc.cpu().numpy()
      pc = PC_pair.PC0.pc
      for hpr_radius in hpr_rads:
        #visible_inds, visible_inds_pc0, visible_inds_pc1 = covisible_inds(PC_pair.PC0, PC_pair.PCUnion, T0, T1, hpr_radius=hpr_radius)
        visible_inds = visible_points(pc, zero_vec, hpr_radius=hpr_radius)
        visible_inds_other = visible_points(pc, viewpoint1_CS0, hpr_radius=hpr_radius)
        # Calculate the percentage of visible points and add to the sum
        sum_visible_perc_own[hpr_radius] += 100*len(visible_inds)/pc.shape[0]
        sum_visible_perc_other[hpr_radius] += 100*len(visible_inds_other)/pc.shape[0]

      # plot_points(pc, viewpoint=zero_vec, plot_viewpoint=True, extra_viewpoint=viewpoint1_CS0, points_special_inds=visible_inds)

# Calculate and print average visibility for each radius
for hpr_radius in hpr_rads:
    avg_visible_perc_own = sum_visible_perc_own[hpr_radius] / len(PC_scenes_training)
    avg_visible_perc_other = sum_visible_perc_other[hpr_radius] / len(PC_scenes_training)
    print(f"Average percentage of visible points for rad {hpr_radius}: {np.around(avg_visible_perc_own, 2)}% vs other {np.around(avg_visible_perc_other, 2)}%")


#plot_points(pc, viewpoint=zero_vec, plot_viewpoint=True, extra_viewpoint=viewpoint1_CS0, points_special_inds=visible_inds)

In [ ]:
%%timeit
visible_inds = visible_points(points, viewpoint, hpr_radius=3.25)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from visualization.point_e_tools import get_point_e_model, scatter_spheres
from visualization.point_e.point_e.util.plotting import plot_point_cloud

from utils.visibility import visible_points

In [ ]:
# Generate synthetic point cloud with point-E (from text to point cloud)
# Takes about 1-2 min to run so don't re-run it if not necessary
samples, sampler = get_point_e_model(text='two fotballs')

In [ ]:
pc = sampler.output_to_point_clouds(samples)[0]
viewpoint = np.array([0.45, -0.4, 0.5])
visible_inds = visible_points(pc.coords, viewpoint, hpr_radius=3.7)
print(pc.coords.shape, type(pc.coords))
#pc.coords = pc.coords[visible_inds]
print(pc.coords.shape, type(pc.coords))


In [ ]:

#pc = sampler.output_to_point_clouds(samples)[0]
size = 0.50
fig = plot_point_cloud(pc, grid_size=1, fixed_bounds=((-size, -size, -size),(size, size, size)), grid_1d=True)
# Get the current Axes3D object
ax = plt.gca()
# Reduce the opacity of the point cloud
axes = fig.get_axes()
for ax in axes:
    for coll in ax.collections:
        coll.set_alpha(0.1)  # Set the alpha to 0.5 or any other value less than 1



# Add dotted lines from the point to the respective axes
#ax = scatter_spheres(x=-0.3, y=-0.4, z=0.3, ax=ax, col="r", size=size)
#ax = scatter_spheres(x=0.45, y=-0.4, z=0.5, ax=ax, col="b", size=size)
plt.show()

In [ ]:
folder = 'images/point_clouds/point_e/'
scale = 1
viewpoint = np.array([scale, scale, scale])
# plot_points(pc.coords, viewpoint, plot_viewpoint=False, out_file=folder + 'fotballs.eps', save=False)
visible_inds = visible_points(pc.coords, viewpoint=viewpoint, hpr_radius=2.1)
plot_points(pc.coords, viewpoint, plot_viewpoint=True, out_file=folder + 'fotballs_visible_colored.eps', save=True,
            points_special_inds=visible_inds)